# IR Assignment 2 - MVP

That notebook will do these exact things:

find XML files
slice a small subset
parse structured patent fields
build a searchable title + abstract text
create a BM25 lexical baseline
accept a query
return ranked patent results
save sample outputs

This directly implements the first rung of our Assignment 1 plan:

document processing
sparse lexical retrieval
early prototype on a subset
sample data and retrieval explanation.

In [ ]:
print("hello")

: 

In [ ]:
%pip install -q pandas lxml tqdm rank-bm25

In [ ]:
from pathlib import Path
import random
import re
import json

import pandas as pd
from lxml import etree
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

#### MVP Control panel

In [ ]:
# set project root automatically if notebook is inside /notebooks
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

#dataset path
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "clef_ip"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
RESULTS_DIR = PROJECT_ROOT / "results" / "demo"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# mvp settings
SUBSET_SIZE = 2000          # start small increase later
USE_RANDOM_SAMPLE = False   # False = first N sorted files, True = random sample
RANDOM_SEED = 42
TOP_K = 10

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR exists:", RAW_DIR.exists())
print("INTERIM_DIR:", INTERIM_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

defines the folder locations

creates output folders if they do not exist

sets the subset size

sets whether the subset is random or just the first N files

### discover XML files and slice the dataset

In [ ]:
xml_files = sorted(RAW_DIR.rglob("*.xml"))

if len(xml_files) == 0:
    raise FileNotFoundError(
        f"No XML files found under {RAW_DIR}. Put your CLEF-IP XML files in data/raw/clef_ip/"
    )

if USE_RANDOM_SAMPLE:
    rng = random.Random(RANDOM_SEED)
    subset_files = rng.sample(xml_files, min(SUBSET_SIZE, len(xml_files)))
else:
    subset_files = xml_files[:min(SUBSET_SIZE, len(xml_files))]

manifest_path = INTERIM_DIR / "mvp_subset_files.txt"
manifest_path.write_text("\n".join(str(p) for p in subset_files), encoding="utf-8")

print(f"Total XML files found: {len(xml_files):,}")
print(f"MVP subset size: {len(subset_files):,}")
print(f"Subset manifest saved to: {manifest_path}")

subset_files[:5]

finds all XML files in the raw data folder

creates the exact subset used for the MVP

saves that subset to mvp_subset_files.txt

### XML parsing helpers

In [ ]:
TAG_SPACE_RE = re.compile(r"\s+")

def clean_join(text_items):
    parts = []
    for item in text_items:
        if item is None:
            continue
        text = str(item).strip()
        if text:
            parts.append(text)
    joined = " ".join(parts)
    return TAG_SPACE_RE.sub(" ", joined).strip()

def first_nonempty(root, xpath_list):
    for xp in xpath_list:
        try:
            result = root.xpath(xp)
            text = clean_join(result) if isinstance(result, list) else str(result).strip()
            if text:
                return text
        except Exception:
            continue
    return ""

def parse_patent_xml(xml_path: Path):
    """Robust patent XML parser. Extracts UCID from root @ucid attribute (consistent with parser.py)."""
    parser = etree.XMLParser(recover=True, huge_tree=True)
    tree = etree.parse(str(xml_path), parser)
    root = tree.getroot()

    # UCID extraction — same priority as parser.py
    ucid = root.get("ucid", "").strip()
    if not ucid:
        ucid_els = root.xpath('//*[@ucid]')
        if ucid_els:
            ucid = ucid_els[0].get("ucid", "").strip()
    if not ucid:
        doc_nums = root.xpath(
            '//*[local-name()="publication-reference"]//*[local-name()="doc-number"]'
        )
        if doc_nums and doc_nums[0].text:
            ucid = doc_nums[0].text.strip()
    if not ucid:
        ucid = xml_path.stem

    title       = first_nonempty(root, ['//*[local-name()="invention-title"]/text()', '//*[local-name()="title"]/text()'])
    abstract    = first_nonempty(root, ['//*[local-name()="abstract"]//text()'])
    claims      = first_nonempty(root, ['//*[local-name()="claims"]//text()', '//*[local-name()="claim"]//text()'])
    description = first_nonempty(root, ['//*[local-name()="description"]//text()'])
    ipc         = first_nonempty(root, ['//*[local-name()="classification-ipc"]//text()'])

    if not any([title, abstract, claims, description]):
        raise ValueError("No usable text extracted from XML")

    return {
        "ucid":        ucid,
        "title":       title,
        "abstract":    abstract,
        "claims":      claims,
        "description": description,
        "ipc":         ipc,
        "source_file": str(xml_path)
    }


defines generic helper functions to extract text from patent XML

uses multiple XPath fallbacks because patent XML can vary

extracts the fields spoken about in Assignment 1:

1. title
2. abstract
3. claims
4. description
5. IPC

### parse the subset into a dataframe

In [ ]:
records = []
failures = []

for xml_path in tqdm(subset_files, desc="parsing xml"):
    try:
        records.append(parse_patent_xml(xml_path))
    except Exception as e:
        failures.append({"file": str(xml_path), "error": repr(e)})

docs_df = pd.DataFrame(records)

if docs_df.empty:
    raise ValueError("Parsing produced no usable patent records.")

docs_df["search_text"] = (
    docs_df["title"].fillna("") + " " + docs_df["abstract"].fillna("")
).str.replace(r"\s+", " ", regex=True).str.strip()

docs_df = docs_df[docs_df["search_text"] != ""].copy()
docs_df = docs_df.drop_duplicates(subset=["ucid"]).reset_index(drop=True)

parsed_output_path = INTERIM_DIR / "parsed_patents_mvp.jsonl"
docs_df.to_json(parsed_output_path, orient="records", lines=True, force_ascii=False)

failures_path = INTERIM_DIR / "parse_failures_mvp.json"
failures_path.write_text(json.dumps(failures, indent=2), encoding="utf-8")

print(f"Parsed records kept: {len(docs_df):,}")
print(f"Parsing failures: {len(failures):,}")
print(f"Parsed records saved to: {parsed_output_path}")
print(f"Failure log saved to: {failures_path}")

docs_df[["ucid", "title", "abstract"]].head(3)


: 

runs the parser over the whole chosen subset

stores the extracted fields in a dataframe

builds the first searchable field:

-- search_text = title + abstract

saves the parsed subset to disk

saves parse failures separately

### inspect one parsed record

In [ ]:
sample_row = docs_df.iloc[0]

print("UCID:")
print(sample_row["ucid"])
print("\nTITLE:")
print(sample_row["title"][:500])
print("\nABSTRACT:")
print(sample_row["abstract"][:1000])
print("\nSEARCH TEXT:")
print(sample_row["search_text"][:1200])



confirms that the XML parsing worked properly


### tokenize the searchable text and build BM25

In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")

def tokenize(text: str):
    return TOKEN_RE.findall(text.lower())

docs_df["tokens"] = docs_df["search_text"].map(tokenize)
docs_df = docs_df[docs_df["tokens"].map(len) > 0].reset_index(drop=True)

corpus_tokens = docs_df["tokens"].tolist()
bm25 = BM25Okapi(corpus_tokens)

print(f"Documents in BM25 corpus: {len(corpus_tokens):,}")
print("Sample tokens from first document:")
print(corpus_tokens[0][:40])

tokenizes the searchable text

builds the in-memory BM25 model

### search functions

In [ ]:
def search_bm25(query_text: str, top_k: int = 10, exclude_ucid: str = None):
    query_tokens = tokenize(query_text)

    if len(query_tokens) == 0:
        raise ValueError("Query contains no usable tokens after tokenization.")

    scores = bm25.get_scores(query_tokens)

    result_df = docs_df[["ucid", "title", "abstract", "ipc", "source_file"]].copy()
    result_df["score"] = scores

    if exclude_ucid is not None:
        result_df = result_df[result_df["ucid"] != exclude_ucid]

    result_df = result_df.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)
    result_df.insert(0, "rank", range(1, len(result_df) + 1))
    return result_df

def search_by_ucid(query_ucid: str, top_k: int = 10):
    row = docs_df.loc[docs_df["ucid"] == query_ucid]

    if row.empty:
        raise KeyError(f"ucid not found: {query_ucid}")

    row = row.iloc[0]
    query_text = f"{row['title']} {row['abstract']}".strip()

    return search_bm25(
        query_text=query_text,
        top_k=top_k,
        exclude_ucid=query_ucid
    )


creates the actual search functions

supports two ways to query:

1. free text query
2. patent-as-query by ucid


### run a free-text query

In [ ]:
demo_query = "wireless power transfer charging device"

demo_results = search_bm25(demo_query, top_k=TOP_K)
demo_results[["rank", "ucid", "score", "title"]]


tests the first end to end search with a free text query

gives the core Assignment 2 outcome:

-> query in

-< ranked list out

### run a patent-as-query search

In [ ]:
demo_topic_ucid = docs_df.iloc[0]["ucid"]

topic_results = search_by_ucid(demo_topic_ucid, top_k=TOP_K)
topic_results[["rank", "ucid", "score", "title"]]


takes one patent from the subset

uses its title + abstract as the search query

returns the top ranked similar patents

### save demo outputs

In [ ]:
demo_results_path = RESULTS_DIR / "demo_free_text_results.csv"
topic_results_path = RESULTS_DIR / "demo_topic_results.csv"

demo_results.to_csv(demo_results_path, index=False)
topic_results.to_csv(topic_results_path, index=False)

print("Saved:")
print(demo_results_path)
print(topic_results_path)


### Once we know the notebook works, we will copy the reusable functions here

src/xml_utils.py

src/search_utils.py

src/evaluate.py

### Next notebook :
notebooks/02_bm25f_fields.ipynb

add claims

create field-aware retrieval

compare BM25 vs BM25F